# RG3 조건부 정상 순위 탐지/오탐 점검

기존std4와 rank75의 같은 .10 내부 정상 임계값 설정 비교. .02 보수적 설정도 기록. 사후 개발용 분석이며 선택한 후보의 새 독립 검증은 아님. 19개 가스·61개 오탐 정상 시각과 점수 기록. 실제 원인 설명이 아니라 모델 판정의 진단이다.


In [1]:
from pathlib import Path
import pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
p=pd.read_csv(t/'rg3_conditional_rank_predictions.csv')
base=p[(p.model=='std4_reference')&(p.budget==.1)]
loose=p[(p.model=='conditional_rank75')&(p.budget==.1)]
strict=p[(p.model=='conditional_rank75')&(p.budget==.02)]
z=loose[['TimeStamp','block','actual','prediction','score','threshold']].rename(columns={'prediction':'rank75_recall_prediction','score':'rank75_score','threshold':'rank75_recall_threshold'})
z=z.merge(base[['TimeStamp','prediction']].rename(columns={'prediction':'std4_recall_prediction'}),on='TimeStamp',validate='one_to_one')
z=z.merge(strict[['TimeStamp','prediction','threshold']].rename(columns={'prediction':'rank75_strict_prediction','threshold':'rank75_strict_threshold'}),on='TimeStamp',validate='one_to_one')
assert len(z)==620 and z.actual.sum()==19
z['new_gas_vs_matched_std4']=(z.actual==1)&(z.rank75_recall_prediction==1)&(z.std4_recall_prediction==0)
z['rank75_normal_false_alarm']=(z.actual==0)&(z.rank75_recall_prediction==1)
z[z.actual==1].to_csv(t/'rg3_conditional_rank_all19_gas_cases.csv',index=False,encoding='utf-8-sig')
z[z.rank75_normal_false_alarm].to_csv(t/'rg3_conditional_rank_false61_cases.csv',index=False,encoding='utf-8-sig')
z.groupby('block').agg(shots=('actual','size'),gas=('actual','sum'),new_gas=('new_gas_vs_matched_std4','sum'),false_alarms=('rank75_normal_false_alarm','sum')).to_csv(t/'rg3_conditional_rank_block_review.csv',encoding='utf-8-sig')
print('New gas vs matched std4:',z.new_gas_vs_matched_std4.sum())
print(z[(z.actual==1)&(z.rank75_recall_prediction==1)][['TimeStamp','block','std4_recall_prediction']].to_string(index=False))
for prefix in ['rg3_conditional_rank','rg3_rank_margin','rg3_rank75_gas_mode','rg3_normal_scaled_rank']:
 pp=pd.read_csv(t/(prefix+'_predictions.csv'));m=pd.read_csv(t/(prefix+'_metrics.csv'))
 a=pd.read_csv(t/(prefix+'_training_audit.csv'))
 assert all(set(str(r.train_blocks).split(',')).isdisjoint(str(r.query_blocks).split(',')) for r in a.itertuples())
 for key,g in pp.groupby(['model','budget']):
  row=m[(m.model==key[0])&(m.budget==key[1])].iloc[0]
  np.testing.assert_array_equal(confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel(),row[['TN','FP','FN','TP']].to_numpy())
  assert len(g)==620 and g.actual.sum()==19
print('All RG3 confusion matrices, coverage and outer/inner block exclusions verified')


New gas vs matched std4: 5
          TimeStamp  block  std4_recall_prediction
2020-10-22 04:46:36      3                       0
2020-10-22 05:10:17      3                       1
2020-10-23 05:15:46      5                       1
2020-10-23 05:59:00      5                       0
2020-10-23 07:15:11      5                       1
2020-10-23 07:21:20      5                       0
2020-11-04 05:17:45      6                       0
2020-11-04 05:18:48      6                       0


All RG3 confusion matrices, coverage and outer/inner block exclusions verified
